In [1]:
import os
import h5py
import requests
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive,userdata
drive.mount("/content/drive")
import scipy.stats as stats
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import time

Mounted at /content/drive


In [16]:
os.chdir("/content/drive/MyDrive/TNGdata2")

In [4]:
baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":userdata.get("TNG_API_KEY")}

In [5]:
def get(path, params=None):
    # make HTTP GET request to path
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename # return the filename string

    return r

In [6]:
r = get(baseUrl)

In [7]:
r['simulations']

[{'name': 'Illustris-1',
  'num_snapshots': 134,
  'url': 'http://www.tng-project.org/api/Illustris-1/'},
 {'name': 'Illustris-1-Dark',
  'num_snapshots': 136,
  'url': 'http://www.tng-project.org/api/Illustris-1-Dark/'},
 {'name': 'Illustris-2',
  'num_snapshots': 136,
  'url': 'http://www.tng-project.org/api/Illustris-2/'},
 {'name': 'Illustris-2-Dark',
  'num_snapshots': 136,
  'url': 'http://www.tng-project.org/api/Illustris-2-Dark/'},
 {'name': 'Illustris-3',
  'num_snapshots': 136,
  'url': 'http://www.tng-project.org/api/Illustris-3/'},
 {'name': 'Illustris-3-Dark',
  'num_snapshots': 136,
  'url': 'http://www.tng-project.org/api/Illustris-3-Dark/'},
 {'name': 'TNG100-1',
  'num_snapshots': 100,
  'url': 'http://www.tng-project.org/api/TNG100-1/'},
 {'name': 'TNG100-1-Dark',
  'num_snapshots': 100,
  'url': 'http://www.tng-project.org/api/TNG100-1-Dark/'},
 {'name': 'TNG100-2',
  'num_snapshots': 100,
  'url': 'http://www.tng-project.org/api/TNG100-2/'},
 {'name': 'TNG100-2-Dark

In [20]:
tng50d1 = get(baseUrl + "TNG50-1")
tng50d2 = get(baseUrl + "TNG50-2")
tng50d3 = get(baseUrl + "TNG50-3")

In [21]:
def get_and_rename(url, params, dest_name):
    temp_filename = get(url, params)
    if os.path.exists(temp_filename) and temp_filename != dest_name:
        if os.path.exists(dest_name):
            os.remove(dest_name)
        os.rename(temp_filename, dest_name)
    return dest_name

subhalo50d1_cutout = get_and_rename(tng50d1['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d1_0.hdf5")
subhalo50d2_cutout = get_and_rename(tng50d2['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d2_0.hdf5")
subhalo50d3_cutout = get_and_rename(tng50d3['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d3_0.hdf5")

In [22]:
cutouts_paths = {
    "1": subhalo50d1_cutout,
    "2": subhalo50d2_cutout,
    "3": subhalo50d3_cutout
}

mag_raw = {}
mag_physical_dict = {}
mag_physical_magnitude_dict = {}
coords_dict = {}
headers_dict = {}

for res in ["1", "2", "3"]:
    cutout_file = cutouts_paths[res]
    with h5py.File(cutout_file, 'r') as f:
        coords_dict[res] = f['PartType0']['Coordinates'][:]
        mag_raw[res] = f['PartType0']['MagneticField'][:]
        headers_dict[res] = dict(f['Header'].attrs)

    h_param = headers_dict[res]['HubbleParam']
    scale_f = 1.0 / (1.0 + headers_dict[res]["Redshift"])

    mag_physical_dict[res] = mag_raw[res] * h_param / scale_f**2 * 2.6e-6

    mag_physical_magnitude_dict[res] = np.sqrt(
        mag_physical_dict[res][:, 0]**2 +
        mag_physical_dict[res][:, 1]**2 +
        mag_physical_dict[res][:, 2]**2
    )

In [23]:
for res in ["1", "2", "3"]:
    m_norm = mag_physical_magnitude_dict[res]
    print(f"Resolution TNG50-{res}:")
    print(f"  Median: {np.median(m_norm):.5e}")
    print(f"  Mean:   {np.mean(m_norm):.5e}")
    print(f"  Min:    {np.min(m_norm):.5e}")
    print(f"  Max:    {np.max(m_norm):.5e}\n")

Resolution TNG50-1:
  Median: 1.15274e-06
  Mean:   5.78380e-06
  Min:    2.21321e-12
  Max:    7.93010e-03

Resolution TNG50-2:
  Median: 8.92205e-07
  Mean:   2.47200e-06
  Min:    1.17414e-12
  Max:    4.10281e-03

Resolution TNG50-3:
  Median: 5.25733e-07
  Mean:   1.60938e-06
  Min:    1.38649e-12
  Max:    7.59466e-05



In [24]:
subhalo_data = {}
for res in ["1", "2", "3"]:
    sim_name = f"TNG50-{res}"
    subhalo_data[res] = get(baseUrl + sim_name + "/snapshots/33/subhalos/0")

sfrs_res0 = {res: subhalo_data[res]['sfr'] for res in ["1", "2", "3"]}
bhmdots_res0 = {res: subhalo_data[res]['bhmdot'] for res in ["1", "2", "3"]}
mstars_res0 = {res: subhalo_data[res]['mass_stars'] for res in ["1", "2", "3"]}
mhalos_res0 = {res: subhalo_data[res]['mass_log_msun'] for res in ["1", "2", "3"]}

for res in ["1", "2", "3"]:
    print(f"Resolution TNG50-{res}: sfr={sfrs_res0[res]:.3f}, bhmdot={bhmdots_res0[res]:.3e}, mstar={mstars_res0[res]:.3f}, mhalo={mhalos_res0[res]:.3f}")

Resolution TNG50-1: sfr=267.727, bhmdot=4.977e-04, mstar=36.256, mhalo=13.307
Resolution TNG50-2: sfr=69.092, bhmdot=1.416e-04, mstar=18.619, mhalo=13.306
Resolution TNG50-3: sfr=28.858, bhmdot=2.175e-04, mstar=16.387, mhalo=13.308


In [30]:
resolutions = ["1", "2", "3"]
num_subhalos = 100
fields = ['sfr', 'bhmdot', 'mass_stars', "mass_bhs", 'mass_log_msun']

data = np.zeros((num_subhalos, len(resolutions), len(fields)+2)) - 1

def get_with_retry(path, params=None, retries=5, backoff=2):
    for attempt in range(retries):
        try:
            return get(path, params)
        except requests.exceptions.HTTPError as e:
            if attempt == retries - 1:
                raise e
            sleep_time = backoff * (2 ** attempt)
            time.sleep(sleep_time)
        except Exception as e:
            if attempt == retries - 1:
                raise e
            time.sleep(backoff)

subhalos_lists = {}
for res in resolutions:
    subhalos_lists[res] = get_with_retry(baseUrl + "TNG50-" + res + "/snapshots/33/subhalos/", {"limit": 100, "order_by": "-mass_stars"})

valid_subhalos = 0
list_idx = -1

while valid_subhalos < num_subhalos:
    list_idx += 1
    if list_idx >= 100:
        for res in resolutions:
            subhalos_lists[res] = get_with_retry(subhalos_lists[res]["next"])
        list_idx = 0

    temp_data = np.zeros((len(resolutions), len(fields)+2)) - 1
    alignment_failed = False
    processed_subhalo_id = None

    for j, res in enumerate(resolutions):
        try:
            subhalo_summary = subhalos_lists[res]["results"][list_idx]
            subhalo = get_with_retry(subhalo_summary["url"])
        except Exception as e:
            print(f"Failed to load subhalo metadata at list index {list_idx} for resolution {res}")
            alignment_failed = True
            break

        if subhalo.get('subhaloflag') != 1:
            alignment_failed = True
            break

        if processed_subhalo_id is None:
            processed_subhalo_id = subhalo['id']

        try:
            subhalo_cutout = get_with_retry(subhalo['cutouts']['subhalo'], {"gas": "Coordinates,MagneticField"})
            with h5py.File(subhalo_cutout, 'r') as f:
                try:
                    mag = f['PartType0']['MagneticField']
                    mag_data = mag[:]
                except KeyError:
                    print(f"No MagneticField data for subhalo {subhalo['id']} at resolution {res}")
                    if os.path.exists(subhalo_cutout):
                        os.remove(subhalo_cutout)
                    alignment_failed = True
                    break
                header = dict(f['Header'].attrs)

            local_hubble = header['HubbleParam']
            local_scale_factor = 1.0 / (1.0 + header['Redshift'])
            mag_physical = mag_data * local_hubble / local_scale_factor**2 * 2.6e-6
            mag_physical_norm = np.sqrt(mag_physical[:,0]**2 + mag_physical[:,1]**2 + mag_physical[:,2]**2)

            for k, field in enumerate(fields):
                value = subhalo.get(field, 0.0)
                if field == "mass_log_msun":
                    value = 10**value
                elif field in ["mass_stars", "mass_bhs"]:
                    value *= 1e10 / local_hubble
                elif field == "bhmdot":
                    value *= 1e10 / 1e9
                temp_data[j, k] = value

            temp_data[j, -2] = np.mean(mag_physical_norm)
            temp_data[j, -1] = np.median(mag_physical_norm)

            if os.path.exists(subhalo_cutout):
                os.remove(subhalo_cutout)

        except Exception as e:
            print(f"Skipping subhalo {subhalo['id']} at resolution {res} due to processing error: {e}")
            if 'subhalo_cutout' in locals() and os.path.exists(subhalo_cutout):
                try:
                    os.remove(subhalo_cutout)
                except Exception:
                    pass
            alignment_failed = True
            break

    if not alignment_failed:
        data[valid_subhalos, :, :] = temp_data
        print(f"Saved aligned subhalo ID {processed_subhalo_id} across all 3 resolutions (Count: {valid_subhalos + 1}/100)")
        valid_subhalos += 1
        if valid_subhalos % 10 == 0:
            np.savez_compressed("subhalo_data.npz", data=data)

Saved aligned subhalo ID 15225 across all 3 resolutions (Count: 1/100)
Saved aligned subhalo ID 0 across all 3 resolutions (Count: 2/100)
Saved aligned subhalo ID 8068 across all 3 resolutions (Count: 3/100)
Saved aligned subhalo ID 21554 across all 3 resolutions (Count: 4/100)
Saved aligned subhalo ID 29443 across all 3 resolutions (Count: 5/100)
Saved aligned subhalo ID 48457 across all 3 resolutions (Count: 6/100)
Saved aligned subhalo ID 46531 across all 3 resolutions (Count: 7/100)
Skipping subhalo 5811 at resolution 2 due to processing error: 503 Server Error: Service Unavailable for url: https://data-eu.tng-project.org/cutout/subhalo/L35n1080TNG/33/5811/16386.0.0.0.0.0/?token=8e37d117c6007262391a
Saved aligned subhalo ID 37157 across all 3 resolutions (Count: 8/100)
Saved aligned subhalo ID 31832 across all 3 resolutions (Count: 9/100)
Saved aligned subhalo ID 42404 across all 3 resolutions (Count: 10/100)
Saved aligned subhalo ID 62458 across all 3 resolutions (Count: 11/100)
S

In [31]:
loaded_data = np.load("subhalo_data.npz")
real_data = loaded_data["data"]
# fields = 'sfr', 'bhmdot', 'mass_stars', 'mass_bhs', 'mass_log_msun'(converted to 10**mass_log_msun) mean and median

for j, res in enumerate(["TNG50-1", "TNG50-2", "TNG50-3"]):
    print(f"\nSummary statistics for {res}:")
    valid_mask = real_data[:, j, 0] != -1
    valid_sfrs = real_data[valid_mask, j, 0]
    valid_mstars = real_data[valid_mask, j, 2]
    valid_b_means = real_data[valid_mask, j, 5]
    valid_b_medians = real_data[valid_mask, j, 6]

    print(f"  Number of valid subhalos: {len(valid_sfrs)}")
    if len(valid_sfrs) > 0:
        print(f"  Mean Stellar Mass: {np.mean(valid_mstars):.3e} Msun")
        print(f"  Mean SFR: {np.mean(valid_sfrs):.3f} Msun/yr")
        print(f"  Mean B (Mean): {np.mean(valid_b_means):.3e} G")
        print(f"  Mean B (Median): {np.mean(valid_b_medians):.3e} G")


Summary statistics for TNG50-1:
  Number of valid subhalos: 100
  Mean Stellar Mass: 1.041e+11 Msun
  Mean SFR: 56.938 Msun/yr
  Mean B (Mean): 2.272e-05 G
  Mean B (Median): 6.854e-06 G

Summary statistics for TNG50-2:
  Number of valid subhalos: 100
  Mean Stellar Mass: 6.869e+10 Msun
  Mean SFR: 28.291 Msun/yr
  Mean B (Mean): 1.547e-05 G
  Mean B (Median): 5.405e-06 G

Summary statistics for TNG50-3:
  Number of valid subhalos: 100
  Mean Stellar Mass: 5.804e+10 Msun
  Mean SFR: 24.221 Msun/yr
  Mean B (Mean): 1.410e-05 G
  Mean B (Median): 4.729e-06 G


In [32]:
# 0: sfr, 1: bhmdot, 2: mass_stars, 3: mass_bhs, 4: total_halo_mass, 5: B_mean, 6: B_median
for res_idx in range(3):
    b_mean = real_data[:, res_idx, 5]
    sfrs = real_data[:, res_idx, 0].copy()
    bhmdots = real_data[:, res_idx, 1].copy()

    N = len(b_mean)
    dof = N - 2

    # Handle zeros in log space by substituting with 0.1 * min non-zero value
    mask_sfrs = sfrs > 0
    mask_bhmdots = bhmdots > 0
    num_zeros_sfrs = np.sum(~mask_sfrs)
    num_zeros_bhmdots = np.sum(~mask_bhmdots)

    if np.sum(mask_sfrs) > 0:
        min_for_zeros_sfrs = 0.1 * np.min(sfrs[mask_sfrs])
    else:
        min_for_zeros_sfrs = 1e-4

    if np.sum(mask_bhmdots) > 0:
        min_for_zeros_bhmdots = 0.1 * np.min(bhmdots[mask_bhmdots])
    else:
        min_for_zeros_bhmdots = 1e-6

    sfrs[~mask_sfrs] = min_for_zeros_sfrs
    bhmdots[~mask_bhmdots] = min_for_zeros_bhmdots

    # Statistics
    rho_sfr, p_rho_sfr = stats.spearmanr(b_mean, sfrs)
    rho_bhmdot, p_rho_bhmdot = stats.spearmanr(b_mean, bhmdots)

    log_bmean = np.log10(b_mean)
    log_sfrs = np.log10(sfrs)
    log_bhmdots = np.log10(bhmdots)

    # Bootstrap Spearman
    bootstrap_spearman_sfr = []
    for i in range(len(sfrs)):
        ind = np.random.choice(len(sfrs), size=len(sfrs), replace=True)
        r, _ = stats.spearmanr(b_mean[ind], sfrs[ind])
        bootstrap_spearman_sfr.append(r)
    lower_sfr_sp = np.percentile(bootstrap_spearman_sfr, 2.5)
    upper_sfr_sp = np.percentile(bootstrap_spearman_sfr, 97.5)

    bootstrap_spearman_bhmdot = []
    for i in range(len(bhmdots)):
        ind = np.random.choice(len(bhmdots), size=len(bhmdots), replace=True)
        r, _ = stats.spearmanr(b_mean[ind], bhmdots[ind])
        bootstrap_spearman_bhmdot.append(r)
    lower_bhmdot_sp = np.percentile(bootstrap_spearman_bhmdot, 2.5)
    upper_bhmdot_sp = np.percentile(bootstrap_spearman_bhmdot, 97.5)

    # Pearson on log quantities
    r_sfr, p_r_sfr = stats.pearsonr(log_bmean, log_sfrs)
    r_bhmdot, p_r_bhmdot = stats.pearsonr(log_bmean, log_bhmdots)

    # 5-Fold Cross Validation
    num_folds = 5
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    bmean_preds_sfr = np.zeros_like(log_bmean)

    for train_ind, val_ind in kf.split(log_sfrs):
        X_train, X_val = log_sfrs[train_ind].reshape(-1, 1), log_sfrs[val_ind].reshape(-1, 1)
        y_train = log_bmean[train_ind]
        model = LinearRegression()
        model.fit(X_train, y_train)
        bmean_preds_sfr[val_ind] = model.predict(X_val)

    rmse_sfr = np.sqrt(mean_squared_error(log_bmean, bmean_preds_sfr))
    r2_sfr = r2_score(log_bmean, bmean_preds_sfr)

    bmean_preds_bhmdots = np.zeros_like(log_bmean)
    for train_ind, val_ind in kf.split(log_bhmdots):
        X_train, X_val = log_bhmdots[train_ind].reshape(-1, 1), log_bhmdots[val_ind].reshape(-1, 1)
        y_train = log_bmean[train_ind]
        model = LinearRegression()
        model.fit(X_train, y_train)
        bmean_preds_bhmdots[val_ind] = model.predict(X_val)

    rmse_bhmdot = np.sqrt(mean_squared_error(log_bmean, bmean_preds_bhmdots))
    r2_bhmdot = r2_score(log_bmean, bmean_preds_bhmdots)

    print(f"\n{'='*20} TNG50-{res_idx+1} {'='*20}")
    print(f"N, d.o.f.: {N}, {dof}")
    print(f"Zeros (SFR, BHMDOT): {num_zeros_sfrs}, {num_zeros_bhmdots}")
    print(f"Spearman rho SFR: {rho_sfr:.4f} (95% CI: [{lower_sfr_sp:.3f}, {upper_sfr_sp:.3f}]), p-val: {p_rho_sfr:.2e}")
    print(f"Spearman rho BHMDOT: {rho_bhmdot:.4f} (95% CI: [{lower_bhmdot_sp:.3f}, {upper_bhmdot_sp:.3f}]), p-val: {p_rho_bhmdot:.2e}")
    print(f"Pearson r SFR (log): {r_sfr:.4f}, p-val: {p_r_sfr:.2e}")
    print(f"Pearson r BHMDOT (log): {r_bhmdot:.4f}, p-val: {p_r_bhmdot:.2e}")
    print(f"Cross-Validation RMSE (SFR, BHMDOT): {rmse_sfr:.4f}, {rmse_bhmdot:.4f}")
    print(f"Cross-Validation R2 (SFR, BHMDOT): {r2_sfr:.4f}, {r2_bhmdot:.4f}")


==================== TNG50-1 ====================
N, d.o.f.: 100, 98
Zeros (SFR, BHMDOT): 0, 4
Spearman rho SFR: 0.4727 (95% CI: [0.300, 0.626]), p-val: 6.86e-07
Spearman rho BHMDOT: 0.4995 (95% CI: [0.343, 0.630]), p-val: 1.22e-07
Pearson r SFR (log): 0.5082, p-val: 6.74e-08
Pearson r BHMDOT (log): 0.4352, p-val: 6.04e-06
Cross-Validation RMSE (SFR, BHMDOT): 0.3914, 0.4080
Cross-Validation R2 (SFR, BHMDOT): 0.2298, 0.1629

==================== TNG50-2 ====================
N, d.o.f.: 100, 98
Zeros (SFR, BHMDOT): 1, 2
Spearman rho SFR: 0.4967 (95% CI: [0.309, 0.647]), p-val: 1.47e-07
Spearman rho BHMDOT: 0.5853 (95% CI: [0.453, 0.707]), p-val: 1.61e-10
Pearson r SFR (log): 0.3839, p-val: 8.06e-05
Pearson r BHMDOT (log): 0.4933, p-val: 1.84e-07
Cross-Validation RMSE (SFR, BHMDOT): 0.4905, 0.4512
Cross-Validation R2 (SFR, BHMDOT): 0.0219, 0.1725

==================== TNG50-3 ====================
N, d.o.f.: 100, 98
Zeros (SFR, BHMDOT): 1, 1
Spearman rho SFR: 0.5080 (95% CI: [0.347, 0.670]